In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd
import torch
import pydicom
import matplotlib.pyplot as plt

from pathlib import Path

# MONAI imports
import monai
from monai.data import Dataset, CacheDataset, DataLoader, PILReader
from monai.transforms import (
    LoadImage, LoadImaged, Resized, Compose, SaveImage, 
    Spacingd, SpatialCropd, ResizeWithPadOrCropd
)

import numpy as np
from monai.transforms import (
    Compose,
    LoadImaged,
    Transposed,
    NormalizeIntensityd,
    MapTransform,
    ScaleIntensityRangePercentilesd,
    RandAffined, RandGaussianNoised, 
    RandStdShiftIntensityd, RandScaleIntensityd, RandAdjustContrastd, RandHistogramShiftd,
    ScaleIntensityd, Lambdad,
    LoadImage, Transpose
)

from torch.utils.data import DataLoader
from tqdm.notebook import tqdm



import landmarker
import landmarker.datasets
from landmarker.datasets import get_cepha_landmark_datasets
from landmarker.heatmap import GaussianHeatmapGenerator, LaplacianHeatmapGenerator
from landmarker.models import OriginalSpatialConfigurationNet
from landmarker.losses import GaussianHeatmapL2Loss
from torch.utils.data import DataLoader
from landmarker.visualize import inspection_plot
from landmarker.visualize.utils import prediction_inspect_plot, prediction_inspect_plot_transposed, inspection_plot_numbers
from landmarker.visualize import detection_report
from landmarker.visualize.evaluation import detection_report, convert_to_report_df, evaluate_model_on_loader
from landmarker.data import LandmarkDataset

#   My stuff
import ra_utils
import ra_utils.data.data_utils
from  ra_utils.data.data_utils import (
    extract_extras_from_filename, 
    extract_extras_from_abspath
)

import ra_utils.visualization.plot_landmarks
import ra_utils.data
import ra_utils.data.data_handler
import ra_utils.data.dataloader_CR_landmarks
import ra_utils.visualization.plot_landmarks #.plot_landmarks
import pydicom
import numpy as np

import pydicom
import numpy as np
import pandas as pd


import numpy as np
from sklearn.model_selection import KFold

from ra_utils.data.splits_utils import generate_split_dictionary

import mlflow
from mlflow.tracking import MlflowClient
import yaml
import mlflow.pytorch

from ra_utils.utils.config_parser import load_config
import os 
from landmarker.heatmap.decoder import heatmap_to_coord
from landmarker.metrics import point_error


from pprint import pprint

In [ ]:
# define mlflow runs directory
mlflow_runs_dir = "/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/" 
os.environ["MLFLOW_TRACKING_URI"] = mlflow_runs_dir

# config = load_config(default_config="/home/cwatzenboeck/code/RA/ra_utils/runs/config_landmarks/train_landmarks_01.yaml", 
#                      debugging_in_jupyter_nb=True)

# run_id = "bb72a1aaf7f9476885d559d7baba9b8e"   # best hands run
run_id = "ba6753effad7435d92b2b1d0cf4c33e6"   # best feet run


logged_model_uri = f"runs:/{run_id}/best_model"  # or the path you used
model = mlflow.pytorch.load_model(logged_model_uri)
model.eval()

logged_heatmap_uri = f"runs:/{run_id}/best_heatmap_generator"  # or the path you used
heatmap_generator = mlflow.pytorch.load_model(logged_heatmap_uri)
heatmap_generator.eval();


#mlflow_runs_dir = "/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/"

#"/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/823469097966318542/6cbfd9effd0a49e3bff2441790f4704e/artifacts"

client = MlflowClient()
local_path = client.download_artifacts(run_id, "config")

# Load YAML config
with open(local_path, 'r') as f:
    config = yaml.safe_load(f)

pprint(config)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = "cpu"
print(device)


In [ ]:
base_dir = Path("/home/cwatzenboeck/data/AutoPIX_cirdata/projects__autoscora/")
# base_dir = Path("/home/clemens/data/AutoPIX_cirdata/")
# base_dir = Path("/home/clemens/data/AutoPIX_cirdata_local/")



dataHandler = ra_utils.data.data_handler.DataHandler_CR_autoscoRA(
                folder_H_images = base_dir / "autoscoRA_images/H_images_of_interest_2_renamed_mirrored_inverted_dicoms",
                folder_F_images = base_dir / "autoscoRA_images/F_images_of_interest_2_renamed_mirrored_inverted_dicoms",
                df_lm_labels_H = base_dir / "landmark_data/100_all_H_joints36/points.csv",
                df_lm_labels_F = base_dir / "landmark_data/100_all_F_joints27/points.csv",
                df_autoscoRA_labels_F = base_dir / "autoscoRA_data/autoscoRA_feet.csv",
                df_autoscoRA_labels_H = base_dir / "autoscoRA_data/autoscoRA_hands.csv",
                #training_test_splits_json_H = base_dir / "landmark_data/splits/splits_H_TD_25-03-05.json",
                #training_test_splits_json_H = base_dir / "landmark_data/splits/splits_H_TD_25-03-06__exclude_images_without_header.json",
                training_test_splits_json_H = base_dir / "landmark_data/splits/splits_H_TD_25-03-05_three_sets.yml",
                training_test_splits_json_F = base_dir / "landmark_data/splits/splits_F_TD_25-03-05.yml",
            )




In [ ]:
# # Maybe add spacing in better way
df = dataHandler.df_images_and_landmarks_F

In [ ]:
# Define transforms 
#from landmarker.transforms.images import UseOnlyFirstChannel

fn_keys = ('image',)
spatial_transformd = [RandAffined(fn_keys, prob=1,
                        rotate_range=(-np.pi/12, np.pi/12),
                        translate_range=(-10, 10),
                        scale_range=(-0.1, 0.1),
                        shear_range=(-0.1, 0.1)
                        )]

train_transformd = Compose([
                            #UseOnlyFirstChannel(('image', )),
                            Transposed(keys=["image"], indices=(0, 2, 1)), # CW added
                            RandGaussianNoised(('image', ), prob=0.2, mean=0, std=0.1),  # Add gaussian noise
                            RandScaleIntensityd(('image', ), factors=0.25, prob=0.2),  # Add random intensity scaling
                            RandAdjustContrastd(('image', ), prob=0.2, gamma=(0.7,2.0)),  # Randomly adjust contrast
                            RandHistogramShiftd(('image', ), prob=0.2),  # Randomly shift histogram
                            ScaleIntensityd(('image', )),  # Scale intensity
                        ] + spatial_transformd)

inference_transformd = Compose([
    #UseOnlyFirstChannel(('image', )),
    Transposed(keys=["image"], indices=(0, 2, 1)),  # CW added
    ScaleIntensityd(('image', )),
])




In [ ]:

dim_image = (512,512)

(
    image_paths_train,
    image_paths_test1,
    image_paths_test2,
    landmarks_train,
    landmarks_test1,
    landmarks_test2,
    pixel_spacings_train,
    pixel_spacings_test1,
    pixel_spacings_test2
) = dataHandler.get_landmarks_dataset_F(get_pixel_spacing=True)

if not isinstance(pixel_spacings_train, (list, tuple, np.ndarray)) or pixel_spacings_train is None:
    pixel_spacings_train = (0.1, 0.1)
if not isinstance(pixel_spacings_test1, (list, tuple, np.ndarray)) or pixel_spacings_test1 is None:
    pixel_spacings_test1 = (0.1, 0.1)
if not isinstance(pixel_spacings_test2, (list, tuple, np.ndarray)) or pixel_spacings_test2 is None:
    pixel_spacings_test2 = (0.1, 0.1)


N_train = len(image_paths_train)
# TODO Only do not load some of the images

ds_train, ds_test1, ds_test2 = ra_utils.data.dataloader_CR_landmarks.get_landmark_datasets(
    image_paths_train = image_paths_train,
    image_paths_test1 = image_paths_test1,
    image_paths_test2 = image_paths_test2,
    landmarks_train = landmarks_train,
    landmarks_test1 = landmarks_test1,
    landmarks_test2 = landmarks_test2,
    pixel_spacings_train=pixel_spacings_train,
    pixel_spacings_test1=pixel_spacings_test1,
    pixel_spacings_test2=pixel_spacings_test2,
    train_transform=train_transformd,
    inference_transform=inference_transformd,
    dim_img=dim_image
    )

N_landmarks = landmarks_train.shape[1]

In [ ]:
len(image_paths_train), len(image_paths_test1), len(image_paths_test2)



In [ ]:
ds_train.img_paths[0]

In [ ]:


# from landmarker.heatmap.generator import GaussianHeatmapGenerator

# heatmap_generator = GaussianHeatmapGenerator(
#     nb_landmarks=N_landmarks,
#     sigmas=3,
#     gamma=100,
#     heatmap_size=dim_image,
#     learnable=True, # If True, the heatmap generator will be trainable
# )

# heatmap_generator = LaplacianHeatmapGenerator(
#     nb_landmarks=N_landmarks,
#     sigmas=3,
#     gamma=10,
#     heatmap_size=dim_image,
#     learnable=True, # If True, the heatmap generator will be trainable
# )

heatmap_generator.to("cpu")



In [ ]:
# inspection_plot_numbers(ds_train, range(1))

In [ ]:
# Plot the first 3 images from the training set
inspection_plot(ds_train, range(4), heatmap_generator=heatmap_generator)



In [ ]:
# Plot the first 3 images from dataset without transforms
#heatmap_generator.device = "cpu" # because dataset tensors are still on cpu
inspection_plot(ds_test1, 0, heatmap_generator=heatmap_generator)
#heatmap_generator.device = device # set the desired device back



In [ ]:
# # Redefine heatmaps and make learnable: 
# heatmap_generator = GaussianHeatmapGenerator(
#     nb_landmarks=N_landmarks,
#     sigmas=torch.tensor(5.0, device=device),
#     gamma=10,
#     heatmap_size=dim_image,
#     learnable=True, # If True, the heatmap generator will be trainable
# )

# heatmap_generator = LaplacianHeatmapGenerator(
#     nb_landmarks=N_landmarks,
#     sigmas=torch.tensor(5.0, device=device),
#     gamma=10,
#     heatmap_size=dim_image,
#     learnable=True, # If True, the heatmap generator will be trainable
# )


### Evaluate model: 

In [ ]:
## Set up dataloaders
batch_size = 1
train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(ds_test1, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(ds_test2, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
# Evaluate model on test set
results = evaluate_model_on_loader(
    model=model,
    test_loader=test_loader,
    device=device
)
pred_landmarks = results["pred_landmarks"]
true_landmarks = results["true_landmarks"]
pixel_spacings = results["pixel_spacings"]
dim_origs = results["dim_origs"]
paddings = results["paddings"]
test_mpes = results["test_mpes"]
test_mpe = test_mpes.mean()

print(f"Test Mean PE: {test_mpe:.4f}")




In [ ]:
worst_prediction = test_mpes.argsort()[:][::-1]
print(worst_prediction)
print(test_mpes[worst_prediction])



In [ ]:
from landmarker.metrics import sdr

sdr_test = sdr([2.0, 2, 3.0, 4.0, 5.0], true_landmarks=true_landmarks, pred_landmarks=pred_landmarks,
               dim=(512, 512), dim_orig=dim_origs.int(), pixel_spacing=pixel_spacings, padding=paddings)
for key in sdr_test:
    print(f"SDR for {key}mm: {sdr_test[key]:.4f}")


    

In [ ]:

model.eval()
model.to("cpu")

# print("Best predictions")
# best_predictions = worst_prediction[-2:]
# prediction_inspect_plot_transposed(ds_test2, model, best_predictions) #ds_test2.indices[:3])

print("Median predictions")
medium_predictions = worst_prediction[len(worst_prediction)//2-1: len(worst_prediction)//2+1]
prediction_inspect_plot_transposed(ds_test2, model, medium_predictions) #ds_test2.indices[:3])


print("Worst predictions")
prediction_inspect_plot_transposed(ds_test2, model, worst_prediction[:2])


In [ ]:
report = detection_report(true_landmarks, pred_landmarks, dim=(512, 512), dim_orig=dim_origs.int(),
                    pixel_spacing=pixel_spacings, padding=paddings, class_names=ds_train.class_names,
                    radius=[2.0, 5.0, 10], digits=2, output_dict=True, print_report=False)



report = convert_to_report_df(report)
report.round(2)

In [ ]:
from landmarker.visualize import plot_cpe

plot_cpe(true_landmarks, pred_landmarks, dim=(512, 512), dim_orig=dim_origs.int(),
                    pixel_spacing=pixel_spacings, padding=paddings, class_names=ds_train.class_names,
                    group=False, title="CPE curve", save_path=None,
                    stat='proportion', unit='mm', kind='ecdf')